# Phase 4 — Pilot run and sanity checks
**Goal:** run a *small* version of the experiment to catch bugs and measure run time **before** we spend many hours of computing on Phase 5.

A pilot is like a rehearsal. If something is wrong (a noise function, the decomposition, a seed), it is much cheaper to find it now on 2 datasets in a few minutes than after a full run.

**This notebook does four things:**
1. Runs a mini-grid: 2 datasets × 6 models × 3 noise types × rates {0 %, 20 %} × 11 rounds
2. Checks six sanity conditions; each must PASS
3. Makes two technical decisions: XGBoost start-up time, and GPU vs CPU
4. Times every model on all 15 datasets and estimates how long Phase 5 will take

Kernel: **Python (labelnoise)**.

In [1]:
import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd

NB_DIR = Path.cwd()
BASE = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
sys.path.insert(0, str(BASE / "src"))
from core import (load_dataset, make_model, reference_probs, inject_noise, run_condition,
                  decompose, MODEL_NAMES, MODEL_FAMILY, NOISE_TYPES)

SEED = json.loads((BASE / "config.json").read_text())["SEED"]
DATA_DIR, RAW_DIR = BASE / "data", BASE / "results" / "raw"
ALL_DATASETS = pd.read_csv(BASE / "results" / "tables" / "datasets.csv")["dataset"].tolist()
print(len(ALL_DATASETS), "datasets:", ", ".join(ALL_DATASETS))

15 datasets: kc2, wdbc, ilpd, blood-transfusion, diabetes, tic-tac-toe, credit-g, qsar-biodeg, banknote-authentication, kc1, kr-vs-kp, spambase, phoneme, PhishingWebsites, electricity


## Step 1 — The pilot grid
**Datasets:** wdbc (easy, 569 rows) and diabetes (hard, naturally noisy, 768 rows): one easy and one hard case.

**Conditions per dataset:** each model is run once on clean labels (0 %) and once for each noise type at 20 %, so 6 models × 4 = 24 conditions per dataset and 48 in total.

**Why only 11 rounds (instead of 51)?** The pilot checks that the code *behaves* correctly, not the final numbers. Eleven rounds are enough to see clear patterns and keep the pilot short. 11 is odd, so the majority vote can't tie.

`run_one()` is the same loop Phase 5 will use: it records every number we need for one condition as one row.

In [2]:
def run_one(dataset, model, noise_type, rho, R, data):
    X_tr, y_tr, X_te, y_te, p_ref = data
    t = time.perf_counter()
    P, stats = run_condition(X_tr, y_tr, X_te, model, noise_type, rho, R=R, seed=SEED, p_ref=p_ref)
    return {"dataset": dataset, "model": model, "family": MODEL_FAMILY[model],
            "noise_type": noise_type, "rho": rho, "R": R,
            **decompose(P, y_te), **stats, "seconds": time.perf_counter() - t}

PILOT_DATASETS, R_PILOT, RHO = ["wdbc", "diabetes"], 11, 0.20
rows = []
t0 = time.perf_counter()
for ds in PILOT_DATASETS:
    X_tr, y_tr, X_te, y_te = load_dataset(ds, DATA_DIR)
    data = (X_tr, y_tr, X_te, y_te, reference_probs(X_tr, y_tr, seed=SEED))
    for model in MODEL_NAMES:
        rows.append(run_one(ds, model, "none", 0.0, R_PILOT, data))
        for nt in NOISE_TYPES:
            rows.append(run_one(ds, model, nt, RHO, R_PILOT, data))
pilot = pd.DataFrame(rows)
pilot.to_csv(RAW_DIR / "_pilot.csv", index=False)
print(f"{len(pilot)} conditions in {time.perf_counter() - t0:.0f} s  ->  saved results/raw/_pilot.csv")

48 conditions in 15 s  ->  saved results/raw/_pilot.csv


## Step 2 — The six sanity checks
Each check tests something we *know* must be true if the code is correct. If one fails, it points to a specific kind of bug:

| # | Check | Expected | If it fails |
|---|---|---|---|
| 1 | Identity | error = bias + unbiased var − biased var on every row | Bug in `decompose()` |
| 2 | Realised noise rate | symmetric and boundary: 20 % of rows; one-sided: 20 % of positives | Bug in the noise functions |
| 3 | Clean variance order | On clean labels, Tree variance > Bagging and RF variance | Models not set up as intended |
| 4 | Noise hurts | Error at 20 % > error at 0 % | Noise applied to the test set, or not applied at all |
| 5 | Boundary targeting | Points flipped by boundary noise are closer to the boundary than average | Bug in the boundary weights |
| 6 | Repeatability | Running the same condition twice gives identical predictions | A random step without a seed |

In [3]:
checks = []
def record(n, name, passed, detail):
    checks.append({"#": n, "check": name, "result": "PASS" if passed else "FAIL", "detail": detail})

# 1 Identity
gap = pilot["identity_gap"].abs().max()
record(1, "Identity", gap < 1e-12, f"largest gap {gap:.1e}")

# 2 Realised noise rate (one-sided: rate is a share of the positives, so compare with 0.2 x class-1 share)
noisy = pilot[pilot.noise_type != "none"].copy()
pos_share = {ds: load_dataset(ds, DATA_DIR)[1].mean() for ds in PILOT_DATASETS}
noisy["expected_flip"] = np.where(noisy.noise_type == "one_sided",
                                  RHO * noisy.dataset.map(pos_share), RHO)
dev = (noisy.flip_rate - noisy.expected_flip).abs().max()
record(2, "Realised noise rate", dev < 0.02, f"largest deviation from target {dev:.3f}")

# 3 Clean variance order (total variance = unbiased + biased)
clean = pilot[pilot.noise_type == "none"].assign(var_total=lambda d: d.var_unbiased + d.var_biased)
v = clean.pivot(index="dataset", columns="model", values="var_total")
ok3 = bool(((v["Tree"] > v["Bagging"]) & (v["Tree"] > v["RF"])).all())
record(3, "Clean variance order", ok3,
       "; ".join(f"{ds}: Tree {v.loc[ds,'Tree']:.3f}, Bagging {v.loc[ds,'Bagging']:.3f}, RF {v.loc[ds,'RF']:.3f}" for ds in v.index))

# 4 Noise hurts
cidx = clean.set_index(["dataset", "model"])
for col in ["error", "bias", "net_var"]:
    noisy["d_" + col] = noisy[col].to_numpy() - cidx.loc[list(zip(noisy.dataset, noisy.model)), col].to_numpy()
n_hurt = int((noisy.d_error > 0).sum())
record(4, "Noise hurts", n_hurt == len(noisy), f"error rose in {n_hurt} of {len(noisy)} noisy conditions")

# 5 Boundary targeting (on diabetes, the harder dataset)
X_tr, y_tr, X_te, y_te = load_dataset("diabetes", DATA_DIR)
p_ref = reference_probs(X_tr, y_tr, seed=SEED)
margin = np.abs(p_ref - 0.5)
yb = inject_noise("boundary", y_tr, RHO, np.random.default_rng(0), p_ref)
m_flip, m_all = margin[yb != y_tr].mean(), margin.mean()
record(5, "Boundary targeting", m_flip < m_all, f"avg margin flipped {m_flip:.3f} vs all rows {m_all:.3f}")

# 6 Repeatability
P1, _ = run_condition(X_tr, y_tr, X_te, "RF", "boundary", RHO, R=11, seed=SEED, p_ref=p_ref)
P2, _ = run_condition(X_tr, y_tr, X_te, "RF", "boundary", RHO, R=11, seed=SEED, p_ref=p_ref)
same = bool(np.array_equal(P1, P2))
record(6, "Repeatability", same, "two identical runs gave identical predictions" if same else "predictions differed")

pd.set_option("display.max_colwidth", 120)
pd.DataFrame(checks)

,#,check,result,detail
0,1,Identity,PASS,largest gap 5.6e-17
1,2,Realised noise rate,PASS,largest deviation from target 0.002
2,3,Clean variance order,PASS,"diabetes: Tree 0.192, Bagging 0.091, RF 0.068; wdbc: Tree 0.062, Bagging 0.029, RF 0.017"
3,4,Noise hurts,FAIL,error rose in 33 of 36 noisy conditions
4,5,Boundary targeting,PASS,avg margin flipped 0.196 vs all rows 0.280
5,6,Repeatability,PASS,two identical runs gave identical predictions


## Investigating check 4 ("noise hurts")
Check 4 failed in a few conditions: error did **not** rise. Before trusting the code, we must find out whether that is a **bug** or a **real effect**. Two possible explanations:
- **Too few rounds:** with only 11 rounds, a very small true increase can be hidden by randomness. On diabetes, one test row equals 0.43 % of the error, so differences under ~0.004 are within one row.
- **A real effect:** on a dataset that already contains natural label mistakes, adding a few more near the boundary may genuinely change little.

The test: re-run each exception with the full **51 rounds** and **three different seeds**. A bug would give a clear, repeatable *decrease* in error. A rounding effect would turn into a consistent *increase*. A real "no effect" would hover around zero.

In [4]:
exceptions = noisy[noisy.d_error <= 0][["dataset", "model", "noise_type", "d_error"]]
print("Exceptions in the 11-round pilot:")
print(exceptions.round(4).to_string(index=False), "\n")

recheck = []
for _, r in exceptions.iterrows():
    X_tr, y_tr, X_te, y_te = load_dataset(r.dataset, DATA_DIR)
    p_ref = reference_probs(X_tr, y_tr, seed=SEED)
    for seed in (42, 7, 123):
        e_clean = decompose(run_condition(X_tr, y_tr, X_te, r.model, "symmetric", 0.0, R=51, seed=seed)[0], y_te)["error"]
        e_noisy = decompose(run_condition(X_tr, y_tr, X_te, r.model, r.noise_type, RHO, R=51, seed=seed, p_ref=p_ref)[0], y_te)["error"]
        recheck.append({"dataset": r.dataset, "model": r.model, "noise_type": r.noise_type, "seed": seed,
                        "error clean": e_clean, "error 20%": e_noisy, "Δ error": e_noisy - e_clean,
                        "Δ in test rows": (e_noisy - e_clean) * len(y_te)})
recheck = pd.DataFrame(recheck)
recheck.round(4)

Exceptions in the 11-round pilot:
 dataset    model noise_type  d_error
diabetes  Bagging   boundary  -0.0008
diabetes AdaBoost  one_sided  -0.0008
diabetes AdaBoost   boundary  -0.0118 



,dataset,model,noise_type,seed,error clean,error 20%,Δ error,Δ in test rows
0,diabetes,Bagging,boundary,42,0.2598,0.2754,0.0156,3.6078
1,diabetes,Bagging,boundary,7,0.2636,0.2758,0.0121,2.8039
2,diabetes,Bagging,boundary,123,0.2619,0.2780,0.0160,3.7059
3,diabetes,AdaBoost,one_sided,42,0.2546,0.2633,0.0087,2.0196
4,diabetes,AdaBoost,one_sided,7,0.2561,0.2663,0.0102,2.3529
5,diabetes,AdaBoost,one_sided,123,0.2579,0.2634,0.0055,1.2745
6,diabetes,AdaBoost,boundary,42,0.2546,0.2599,0.0053,1.2353
7,diabetes,AdaBoost,boundary,7,0.2561,0.2518,-0.0043,-1.0000
8,diabetes,AdaBoost,boundary,123,0.2579,0.2602,0.0023,0.5294


**Conclusion:** no bug.
- With 51 rounds, most exceptions turn into a **consistent increase** in error across all seeds. The 11-round pilot was just too imprecise to show it.
- Any case that still hovers around zero does so by **about one test row** in both directions, which is a genuine "almost no effect". That is plausible on a dataset that already contains natural label mistakes near its boundary. Our boundary noise then mostly re-flips points that were already ambiguous.

This is itself worth remembering for the paper: **noise does not always hurt**, and how much it hurts depends on the dataset. That is exactly why we use 15 datasets and statistical tests instead of trusting one example. It also shows why Phase 5 uses 51 rounds, not 11.

## A first look at the pilot numbers
Change in error, bias and net variance caused by 20 % noise (noisy minus clean), averaged over the two pilot datasets. With 2 datasets and 11 rounds this is a **rehearsal, not a result**. It only shows that the output has the shape the paper needs.

In [5]:
(noisy.groupby(["noise_type", "model"], sort=False)[["d_error", "d_bias", "d_net_var"]]
      .mean().round(3).unstack(0).reindex(MODEL_NAMES))

d_error                       d_bias                     \
noise_type symmetric one_sided boundary symmetric one_sided boundary   
model                                                                  
Tree           0.106     0.040    0.058     0.029     0.018    0.043   
Bagging        0.029     0.012    0.015     0.001     0.009   -0.004   
RF             0.027     0.014    0.015     0.015     0.012    0.004   
AdaBoost       0.046     0.034    0.024     0.014     0.023    0.004   
GB             0.042     0.025    0.022     0.004     0.022    0.021   
XGB            0.049     0.030    0.035     0.008     0.015    0.023   

           d_net_var                     
noise_type symmetric one_sided boundary  
model                                    
Tree           0.077     0.022    0.015  
Bagging        0.028     0.004    0.018  
RF             0.011     0.002    0.012  
AdaBoost       0.032     0.011    0.020  
GB             0.039     0.003    0.001  
XGB            0.041     0.014    0.012

## Step 3 — Two technical decisions

**Decision A: XGBoost start-up.** In Phase 0, XGBoost looked slow (2.1 s per fit). Timing three fits in a row shows that only the **first** fit is slow (a one-time start-up cost); later fits are fast. Nothing to fix.

**Decision B: GPU or CPU?** Your PC has an NVIDIA RTX 2050. Only XGBoost can use it (the scikit-learn models are CPU-only). We compare speed **and** results on the largest dataset (electricity). We use the GPU only if it is clearly faster *and* gives the same predictions. Otherwise the CPU keeps all models on the same hardware and the results exactly repeatable.

In [6]:
import warnings
from xgboost import XGBClassifier
X_tr, y_tr, X_te, y_te = load_dataset("electricity", DATA_DIR)

def time_fits(make, n=3):
    times, acc = [], None
    for _ in range(n):
        m = make(); t = time.perf_counter(); m.fit(X_tr, y_tr); times.append(time.perf_counter() - t)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            acc = m.score(X_te, y_te)
    return times, acc

cpu_t, cpu_acc = time_fits(lambda: make_model("XGB", 0))
print("CPU (1 thread):", [f"{x:.3f}s" for x in cpu_t], " test accuracy", round(cpu_acc, 4))
try:
    gpu_t, gpu_acc = time_fits(lambda: XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                                                    tree_method="hist", device="cuda", random_state=0,
                                                    eval_metric="logloss"))
    print("GPU (CUDA):    ", [f"{x:.3f}s" for x in gpu_t], " test accuracy", round(gpu_acc, 4))
    print("\nDecision:", "use GPU" if (np.median(gpu_t) < 0.5 * np.median(cpu_t) and gpu_acc == cpu_acc)
          else "use CPU: the GPU is not clearly faster and/or gives different predictions")
except Exception as e:
    print("GPU not usable here:", type(e).__name__, "-> use CPU")

CPU (1 thread): ['0.333s', '0.187s', '0.172s']  test accuracy 0.8452


GPU (CUDA):     ['0.207s', '0.101s', '0.097s']  test accuracy 0.8425

Decision: use CPU: the GPU is not clearly faster and/or gives different predictions


## Step 4 — Time every model on all 15 datasets and estimate Phase 5
For each dataset and model we run **one batch of 12 rounds** (one round per CPU thread) on clean labels and measure the wall time.

Phase 5 needs 51 rounds = **5 batches** of 12 (the last one partly full) for each of **13 conditions** (clean + 3 noise types × 4 rates). So:

  estimated time = batch time × 5 × 13

Noisy conditions take about as long as clean ones (same data size), so this is a fair estimate. It is an *upper bound*, because the last batch is only partly full.

In [7]:
timing = []
t0 = time.perf_counter()
for ds in ALL_DATASETS:
    X_tr, y_tr, X_te, y_te = load_dataset(ds, DATA_DIR)
    for model in MODEL_NAMES:
        t = time.perf_counter()
        run_condition(X_tr, y_tr, X_te, model, "symmetric", 0.0, R=12, seed=SEED)
        timing.append({"dataset": ds, "rows": len(y_tr) + len(y_te), "model": model,
                       "batch_seconds": time.perf_counter() - t})
timing = pd.DataFrame(timing)
timing["phase5_minutes"] = timing.batch_seconds * 5 * 13 / 60
timing.to_csv(RAW_DIR / "_timing.csv", index=False)
print(f"timing run took {time.perf_counter() - t0:.0f} s")

per_ds = timing.pivot(index="dataset", columns="model", values="phase5_minutes")[MODEL_NAMES]
per_ds["total (min)"] = per_ds.sum(axis=1)
per_ds = per_ds.loc[ALL_DATASETS].round(1)
per_ds

timing run took 75 s


model,Tree,Bagging,RF,AdaBoost,GB,XGB,total (min)
dataset,,,,,,,
kc2,0.0,0.5,0.3,0.3,0.3,0.1,1.4
wdbc,0.0,0.6,0.3,0.4,0.5,0.1,2.0
ilpd,0.0,0.4,0.3,0.3,0.2,0.0,1.3
blood-transfusion,0.0,0.4,0.3,0.3,0.2,0.0,1.2
diabetes,0.0,0.5,0.3,0.4,0.3,0.0,1.4
tic-tac-toe,0.0,0.4,0.3,0.4,0.2,0.0,1.4
credit-g,0.0,0.7,0.3,0.4,0.3,0.1,1.8
qsar-biodeg,0.0,0.9,0.4,0.5,0.6,0.1,2.5
banknote-authentication,0.0,0.4,0.3,0.4,0.3,0.1,1.5


In [8]:
total_h = per_ds["total (min)"].sum() / 60
print(f"Estimated Phase 5 run time with R = 51: {total_h:.1f} hours")
print("Slowest dataset:", per_ds["total (min)"].idxmax(), f"({per_ds['total (min)'].max():.0f} min)")
print("Slowest model overall:", timing.groupby("model").phase5_minutes.sum().idxmax())

Estimated Phase 5 run time with R = 51: 1.4 hours
Slowest dataset: electricity (38 min)
Slowest model overall: Bagging


## What the timing tells us
- The full Phase 5 grid takes only about **1–2 hours** on this PC, far less than the earlier rough guess. Running one model per CPU thread made a big difference.
- **electricity** (45,000 rows) takes the most time, and **Bagging** is the slowest model, as expected: 100 full-depth trees on every bootstrap sample.
- Because time is not a constraint, Phase 5 could use **more rounds** (e.g. R = 101 instead of 51). More rounds make the *main prediction* more stable for borderline test points, which are exactly the points boundary noise targets. Cost: about twice the run time. This is a choice to make at the start of Phase 5.

## Phase 4 checklist
- [ ] Checks 1, 2, 3, 5, 6 print **PASS**; any check-4 exceptions investigated and explained (no bug)
- [ ] GPU vs CPU decision recorded
- [ ] Phase 5 run-time estimate written down

If all checks pass, the code is ready for the full experiment. **Next: Phase 5**, the main experiments (RQ1 and RQ2).